In [ ]:
import sys
import os
import gc
import json
import time
import importlib
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    MarianMTModel,
    MarianTokenizer,
)

TARGET_PATH = "/content/drive/MyDrive/Thesis/models"

if TARGET_PATH not in sys.path:
    sys.path.insert(0, TARGET_PATH)

importlib.invalidate_caches()

from utils import DATA_DIR, MODELS_DIR, EKMAN_LABELS, set_all_seeds
from evaluate import compute_metrics

from train_phase3 import (
    GoEmotionsDataset,
    hf_compute_metrics,
    load_data,
    MODEL_NAME,
    MAX_LENGTH,
    BATCH_SIZE,
    N_EPOCHS,
    LEARNING_RATE,
    WEIGHT_DECAY,
    WARMUP_STEPS,
    NUM_LABELS,
)

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Thesis/results/phase4_bt_control_runs"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SEEDS = [42, 0, 1]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def make_json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    return value


def get_selected_minority_classes(
    train_df,
    top_k=3,
):
    class_counts = (
        train_df["ekman_id"]
        .value_counts()
        .sort_values(ascending=True)
    )

    selected_class_ids = (
        class_counts
        .index[:top_k]
        .astype(int)
        .tolist()
    )

    return selected_class_ids, class_counts


def prepare_original_rows(train_df):
    train_df = train_df.copy()

    train_df["augmentation"] = "original"
    train_df["source_text"] = train_df["text_clean"]
    train_df["source_id"] = (
        train_df["id"]
        if "id" in train_df.columns
        else np.arange(len(train_df))
    )
    train_df["paraphrase_number"] = 0
    train_df["semantic_similarity"] = np.nan
    train_df["passed_similarity_filter"] = np.nan

    return train_df


def save_confusion_matrix_to_folder(
    y_true,
    y_pred,
    labels,
    output_path,
    title,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(labels))),
    )

    figure, axis = plt.subplots(
        figsize=(9, 8)
    )

    image = axis.imshow(
        matrix,
        interpolation="nearest",
        cmap="Blues",
    )

    figure.colorbar(
        image,
        ax=axis,
    )

    axis.set(
        xticks=np.arange(len(labels)),
        yticks=np.arange(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        xlabel="Predicted label",
        ylabel="True label",
        title=title,
    )

    plt.setp(
        axis.get_xticklabels(),
        rotation=45,
        ha="right",
        rotation_mode="anchor",
    )

    threshold = matrix.max() / 2 if matrix.size else 0

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            axis.text(
                column_index,
                row_index,
                format(matrix[row_index, column_index], "d"),
                ha="center",
                va="center",
                color=(
                    "white"
                    if matrix[row_index, column_index] > threshold
                    else "black"
                ),
            )

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)

    return matrix

In [ ]:
def augment_direct_duplication(
    train_df,
    top_k=3,
    copies_per_source=3,
    seed=42,
):
    selected_class_ids, class_counts = get_selected_minority_classes(
        train_df,
        top_k=top_k,
    )

    selected_df = (
        train_df[
            train_df["ekman_id"].isin(selected_class_ids)
        ]
        .copy()
        .reset_index(drop=True)
    )

    extra_rows = []

    for row_index, row in selected_df.iterrows():
        source_id = (
            row["id"]
            if "id" in selected_df.columns
            else row_index
        )

        for copy_number in range(1, copies_per_source + 1):
            extra_rows.append({
                "text_clean": row["text_clean"],
                "ekman_id": int(row["ekman_id"]),
                "augmentation": "direct_duplication",
                "source_text": row["text_clean"],
                "source_id": source_id,
                "paraphrase_number": copy_number,
                "semantic_similarity": 1.0,
                "passed_similarity_filter": True,
            })

    extra_df = pd.DataFrame(extra_rows)

    augmented_df = pd.concat(
        [
            prepare_original_rows(train_df),
            extra_df,
        ],
        ignore_index=True,
    )

    augmented_counts = (
        augmented_df["ekman_id"]
        .value_counts()
        .sort_index()
    )

    report = {
        "method": "direct_duplication",
        "top_k": top_k,
        "copies_per_source": copies_per_source,
        "selected_class_ids": selected_class_ids,
        "original_training_size": int(len(train_df)),
        "selected_source_examples": int(len(selected_df)),
        "generated_synthetic_examples": int(len(extra_df)),
        "augmented_training_size": int(len(augmented_df)),
        "original_class_counts": {
            str(int(class_id)): int(count)
            for class_id, count
            in class_counts.sort_index().items()
        },
        "augmented_class_counts": {
            str(int(class_id)): int(count)
            for class_id, count
            in augmented_counts.items()
        },
    }

    return augmented_df, extra_df, report

In [ ]:
def augment_random_oversampling(
    train_df,
    top_k=3,
    oversample_factor=3,
    seed=42,
):
    rng = np.random.default_rng(seed)

    selected_class_ids, class_counts = get_selected_minority_classes(
        train_df,
        top_k=top_k,
    )

    extra_rows = []

    for class_id in selected_class_ids:
        class_df = (
            train_df[
                train_df["ekman_id"] == class_id
            ]
            .copy()
            .reset_index(drop=True)
        )

        n_to_add = len(class_df) * oversample_factor

        sampled_indices = rng.integers(
            low=0,
            high=len(class_df),
            size=n_to_add,
        )

        sampled_df = class_df.iloc[
            sampled_indices
        ].reset_index(drop=True)

        for sample_number, (_, row) in enumerate(
            sampled_df.iterrows(),
            start=1,
        ):
            source_id = (
                row["id"]
                if "id" in sampled_df.columns
                else sample_number
            )

            extra_rows.append({
                "text_clean": row["text_clean"],
                "ekman_id": int(row["ekman_id"]),
                "augmentation": "random_oversampling",
                "source_text": row["text_clean"],
                "source_id": source_id,
                "paraphrase_number": sample_number,
                "semantic_similarity": 1.0,
                "passed_similarity_filter": True,
            })

    extra_df = pd.DataFrame(extra_rows)

    augmented_df = pd.concat(
        [
            prepare_original_rows(train_df),
            extra_df,
        ],
        ignore_index=True,
    )

    augmented_counts = (
        augmented_df["ekman_id"]
        .value_counts()
        .sort_index()
    )

    report = {
        "method": "random_oversampling",
        "top_k": top_k,
        "oversample_factor": oversample_factor,
        "selected_class_ids": selected_class_ids,
        "original_training_size": int(len(train_df)),
        "generated_synthetic_examples": int(len(extra_df)),
        "augmented_training_size": int(len(augmented_df)),
        "original_class_counts": {
            str(int(class_id)): int(count)
            for class_id, count
            in class_counts.sort_index().items()
        },
        "augmented_class_counts": {
            str(int(class_id)): int(count)
            for class_id, count
            in augmented_counts.items()
        },
    }

    return augmented_df, extra_df, report

In [ ]:
def translate_deterministically(
    texts,
    tokenizer,
    model,
    device,
    batch_size=8,
    max_length=128,
):
    translations = []

    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]

            encoded = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            ).to(device)

            generated = model.generate(
                **encoded,
                max_length=max_length,
                num_beams=4,
                do_sample=False,
            )

            translations.extend(
                tokenizer.batch_decode(
                    generated,
                    skip_special_tokens=True,
                )
            )

    return translations


def generate_reverse_translation_candidates(
    pivot_texts,
    original_texts,
    tokenizer,
    model,
    device,
    candidates_per_source=12,
    batch_size=8,
    max_length=128,
    seed=42,
    maximum_attempts=8,
):
    """
    Generate a pool of distinct English back-translation candidates
    for each source example.
    """
    candidate_groups = []

    temperature_schedule = [
        0.80,
        0.90,
        1.00,
        1.10,
        1.20,
        1.30,
        1.40,
        1.50,
    ]

    top_p_schedule = [
        0.90,
        0.93,
        0.95,
        0.96,
        0.97,
        0.98,
        0.98,
        0.99,
    ]

    with torch.no_grad():
        for start in range(
            0,
            len(pivot_texts),
            batch_size,
        ):
            pivot_batch = pivot_texts[
                start:start + batch_size
            ]

            original_batch = original_texts[
                start:start + batch_size
            ]

            pools = [
                []
                for _ in pivot_batch
            ]

            for attempt in range(maximum_attempts):
                if all(
                    len(pool) >= candidates_per_source
                    for pool in pools
                ):
                    break

                current_seed = seed + start * 100 + attempt

                torch.manual_seed(current_seed)

                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(current_seed)

                temperature = temperature_schedule[
                    min(
                        attempt,
                        len(temperature_schedule) - 1,
                    )
                ]

                top_p = top_p_schedule[
                    min(
                        attempt,
                        len(top_p_schedule) - 1,
                    )
                ]

                encoded = tokenizer(
                    pivot_batch,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                ).to(device)

                generated = model.generate(
                    **encoded,
                    max_length=max_length,
                    do_sample=True,
                    num_beams=1,
                    top_p=top_p,
                    temperature=temperature,
                    num_return_sequences=candidates_per_source,
                )

                decoded = tokenizer.batch_decode(
                    generated,
                    skip_special_tokens=True,
                )

                for local_index, original_text in enumerate(
                    original_batch
                ):
                    start_index = (
                        local_index
                        * candidates_per_source
                    )

                    end_index = (
                        start_index
                        + candidates_per_source
                    )

                    raw_candidates = decoded[
                        start_index:end_index
                    ]

                    normalized_original = (
                        str(original_text)
                        .strip()
                        .lower()
                    )

                    for candidate in raw_candidates:
                        candidate = str(candidate).strip()

                        if not candidate:
                            continue

                        if candidate.lower() == normalized_original:
                            continue

                        if candidate not in pools[local_index]:
                            pools[local_index].append(candidate)

                        if len(pools[local_index]) >= candidates_per_source:
                            break

            candidate_groups.extend(pools)

    return candidate_groups


def create_backtranslation_candidate_pools(
    selected_df,
    pivot="de",
    candidates_per_source=12,
    batch_size=8,
    seed=42,
    maximum_attempts=8,
):
    source_texts = (
        selected_df["text_clean"]
        .astype(str)
        .tolist()
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    forward_name = f"Helsinki-NLP/opus-mt-en-{pivot}"
    reverse_name = f"Helsinki-NLP/opus-mt-{pivot}-en"

    print(f"\nLoading {forward_name}...")

    forward_tokenizer = MarianTokenizer.from_pretrained(
        forward_name
    )

    forward_model = (
        MarianMTModel
        .from_pretrained(forward_name)
        .to(device)
        .eval()
    )

    print("\nTranslating English to pivot language...")

    pivot_texts = translate_deterministically(
        texts=source_texts,
        tokenizer=forward_tokenizer,
        model=forward_model,
        device=device,
        batch_size=batch_size,
    )

    del forward_model
    del forward_tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"\nLoading {reverse_name}...")

    reverse_tokenizer = MarianTokenizer.from_pretrained(
        reverse_name
    )

    reverse_model = (
        MarianMTModel
        .from_pretrained(reverse_name)
        .to(device)
        .eval()
    )

    print("\nGenerating English back-translation candidates...")

    candidate_groups = generate_reverse_translation_candidates(
        pivot_texts=pivot_texts,
        original_texts=source_texts,
        tokenizer=reverse_tokenizer,
        model=reverse_model,
        device=device,
        candidates_per_source=candidates_per_source,
        batch_size=batch_size,
        seed=seed,
        maximum_attempts=maximum_attempts,
    )

    del reverse_model
    del reverse_tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return source_texts, candidate_groups

In [ ]:
def augment_backtranslation_basic(
    train_df,
    top_k=3,
    aug_factor=1,
    candidates_per_source=12,
    maximum_attempts=8,
    pivot="de",
    batch_size=8,
    seed=42,
    method_name=None,
):
    if method_name is None:
        method_name = f"backtranslation_x{aug_factor}"

    selected_class_ids, class_counts = get_selected_minority_classes(
        train_df,
        top_k=top_k,
    )

    selected_df = (
        train_df[
            train_df["ekman_id"].isin(selected_class_ids)
        ]
        .copy()
        .reset_index(drop=True)
    )

    source_texts, candidate_groups = create_backtranslation_candidate_pools(
        selected_df=selected_df,
        pivot=pivot,
        candidates_per_source=candidates_per_source,
        batch_size=batch_size,
        seed=seed,
        maximum_attempts=maximum_attempts,
    )

    extra_rows = []

    for row_index, candidates in enumerate(candidate_groups):
        if len(candidates) < aug_factor:
            raise RuntimeError(
                f"Source row {row_index} produced only "
                f"{len(candidates)} candidates, but aug_factor="
                f"{aug_factor} was requested."
            )

        row = selected_df.iloc[row_index]

        source_id = (
            row["id"]
            if "id" in selected_df.columns
            else row_index
        )

        selected_candidates = candidates[:aug_factor]

        for paraphrase_number, paraphrase in enumerate(
            selected_candidates,
            start=1,
        ):
            extra_rows.append({
                "text_clean": paraphrase,
                "ekman_id": int(row["ekman_id"]),
                "augmentation": method_name,
                "source_text": source_texts[row_index],
                "source_id": source_id,
                "paraphrase_number": paraphrase_number,
                "semantic_similarity": np.nan,
                "passed_similarity_filter": np.nan,
            })

    extra_df = pd.DataFrame(extra_rows)

    augmented_df = pd.concat(
        [
            prepare_original_rows(train_df),
            extra_df,
        ],
        ignore_index=True,
    )

    augmented_counts = (
        augmented_df["ekman_id"]
        .value_counts()
        .sort_index()
    )

    report = {
        "method": method_name,
        "pivot_language": pivot,
        "top_k": top_k,
        "aug_factor": aug_factor,
        "candidates_per_source": candidates_per_source,
        "maximum_attempts": maximum_attempts,
        "selected_class_ids": selected_class_ids,
        "original_training_size": int(len(train_df)),
        "selected_source_examples": int(len(selected_df)),
        "generated_synthetic_examples": int(len(extra_df)),
        "augmented_training_size": int(len(augmented_df)),
        "original_class_counts": {
            str(int(class_id)): int(count)
            for class_id, count
            in class_counts.sort_index().items()
        },
        "augmented_class_counts": {
            str(int(class_id)): int(count)
            for class_id, count
            in augmented_counts.items()
        },
    }

    return augmented_df, extra_df, report

In [ ]:
def compute_pairwise_semantic_similarity(
    original_texts,
    candidate_texts,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    batch_size=64,
):
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as error:
        raise ImportError(
            "sentence-transformers is required for semantic "
            "similarity filtering. Install it with:\n"
            "!pip install -q sentence-transformers"
        ) from error

    model = SentenceTransformer(model_name)

    original_embeddings = model.encode(
        original_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    candidate_embeddings = model.encode(
        candidate_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    similarities = np.sum(
        original_embeddings * candidate_embeddings,
        axis=1,
    )

    return similarities


def augment_backtranslation_similarity_filtered(
    train_df,
    top_k=3,
    aug_factor=3,
    candidates_per_source=12,
    maximum_attempts=8,
    pivot="de",
    batch_size=8,
    seed=42,
    min_similarity=0.75,
    max_similarity=0.98,
    fallback_to_best=True,
    similarity_model_name="sentence-transformers/all-MiniLM-L6-v2",
):

    selected_class_ids, class_counts = get_selected_minority_classes(
        train_df,
        top_k=top_k,
    )

    selected_df = (
        train_df[
            train_df["ekman_id"].isin(selected_class_ids)
        ]
        .copy()
        .reset_index(drop=True)
    )

    source_texts, candidate_groups = create_backtranslation_candidate_pools(
        selected_df=selected_df,
        pivot=pivot,
        candidates_per_source=candidates_per_source,
        batch_size=batch_size,
        seed=seed,
        maximum_attempts=maximum_attempts,
    )

    flat_originals = []
    flat_candidates = []
    flat_source_indices = []

    for source_index, candidates in enumerate(candidate_groups):
        for candidate in candidates:
            flat_originals.append(source_texts[source_index])
            flat_candidates.append(candidate)
            flat_source_indices.append(source_index)

    if len(flat_candidates) == 0:
        raise RuntimeError(
            "No back-translation candidates were generated."
        )

    print("\nComputing semantic similarities...")

    similarities = compute_pairwise_semantic_similarity(
        original_texts=flat_originals,
        candidate_texts=flat_candidates,
        model_name=similarity_model_name,
    )

    candidates_by_source = {
        source_index: []
        for source_index in range(len(source_texts))
    }

    for candidate, source_index, similarity in zip(
        flat_candidates,
        flat_source_indices,
        similarities,
    ):
        passed_filter = (
            similarity >= min_similarity
            and similarity <= max_similarity
        )

        candidates_by_source[source_index].append({
            "text": candidate,
            "similarity": float(similarity),
            "passed_filter": bool(passed_filter),
        })

    extra_rows = []
    insufficient_sources = 0

    for source_index in range(len(source_texts)):
        row = selected_df.iloc[source_index]

        source_id = (
            row["id"]
            if "id" in selected_df.columns
            else source_index
        )

        candidates = candidates_by_source[source_index]

        passing_candidates = [
            candidate
            for candidate in candidates
            if candidate["passed_filter"]
        ]

        passing_candidates = sorted(
            passing_candidates,
            key=lambda item: item["similarity"],
            reverse=True,
        )

        chosen = passing_candidates[:aug_factor]

        if len(chosen) < aug_factor:
            insufficient_sources += 1

            if fallback_to_best:
                already_chosen_texts = {
                    item["text"]
                    for item in chosen
                }

                remaining_candidates = [
                    candidate
                    for candidate in candidates
                    if candidate["text"] not in already_chosen_texts
                ]

                remaining_candidates = sorted(
                    remaining_candidates,
                    key=lambda item: item["similarity"],
                    reverse=True,
                )

                needed = aug_factor - len(chosen)

                chosen.extend(
                    remaining_candidates[:needed]
                )

        for paraphrase_number, candidate in enumerate(
            chosen,
            start=1,
        ):
            extra_rows.append({
                "text_clean": candidate["text"],
                "ekman_id": int(row["ekman_id"]),
                "augmentation": "backtranslation_similarity_filtered",
                "source_text": source_texts[source_index],
                "source_id": source_id,
                "paraphrase_number": paraphrase_number,
                "semantic_similarity": candidate["similarity"],
                "passed_similarity_filter": candidate["passed_filter"],
            })

    extra_df = pd.DataFrame(extra_rows)

    augmented_df = pd.concat(
        [
            prepare_original_rows(train_df),
            extra_df,
        ],
        ignore_index=True,
    )

    augmented_counts = (
        augmented_df["ekman_id"]
        .value_counts()
        .sort_index()
    )

    if len(extra_df) > 0:
        mean_similarity = float(
            extra_df["semantic_similarity"].mean()
        )

        passed_count = int(
            extra_df["passed_similarity_filter"]
            .fillna(False)
            .sum()
        )
    else:
        mean_similarity = None
        passed_count = 0

    report = {
        "method": "backtranslation_similarity_filtered",
        "pivot_language": pivot,
        "top_k": top_k,
        "aug_factor": aug_factor,
        "candidates_per_source": candidates_per_source,
        "maximum_attempts": maximum_attempts,
        "min_similarity": min_similarity,
        "max_similarity": max_similarity,
        "fallback_to_best": fallback_to_best,
        "similarity_model_name": similarity_model_name,
        "selected_class_ids": selected_class_ids,
        "original_training_size": int(len(train_df)),
        "selected_source_examples": int(len(selected_df)),
        "generated_synthetic_examples": int(len(extra_df)),
        "sources_with_insufficient_passing_candidates": int(insufficient_sources),
        "passed_similarity_filter_count": passed_count,
        "mean_semantic_similarity": mean_similarity,
        "augmented_training_size": int(len(augmented_df)),
        "original_class_counts": {
            str(int(class_id)): int(count)
            for class_id, count
            in class_counts.sort_index().items()
        },
        "augmented_class_counts": {
            str(int(class_id)): int(count)
            for class_id, count
            in augmented_counts.items()
        },
    }

    return augmented_df, extra_df, report

In [ ]:
def apply_augmentation_control(
    train_df,
    control_name,
    seed=42,
    top_k=3,
):
    if control_name == "direct_duplication_x3":
        return augment_direct_duplication(
            train_df=train_df,
            top_k=top_k,
            copies_per_source=3,
            seed=seed,
        )

    if control_name == "random_oversampling_x3":
        return augment_random_oversampling(
            train_df=train_df,
            top_k=top_k,
            oversample_factor=3,
            seed=seed,
        )

    if control_name == "bt_single_x1":
        return augment_backtranslation_basic(
            train_df=train_df,
            top_k=top_k,
            aug_factor=1,
            candidates_per_source=8,
            maximum_attempts=8,
            pivot="de",
            batch_size=8,
            seed=seed,
            method_name="backtranslation_single_x1",
        )

    if control_name == "bt_diverse_x2":
        return augment_backtranslation_basic(
            train_df=train_df,
            top_k=top_k,
            aug_factor=2,
            candidates_per_source=10,
            maximum_attempts=8,
            pivot="de",
            batch_size=8,
            seed=seed,
            method_name="backtranslation_diverse_x2",
        )

    if control_name == "bt_diverse_x3":
        return augment_backtranslation_basic(
            train_df=train_df,
            top_k=top_k,
            aug_factor=3,
            candidates_per_source=12,
            maximum_attempts=8,
            pivot="de",
            batch_size=8,
            seed=seed,
            method_name="backtranslation_diverse_x3",
        )

    if control_name == "bt_similarity_filtered_x3":
        return augment_backtranslation_similarity_filtered(
            train_df=train_df,
            top_k=top_k,
            aug_factor=3,
            candidates_per_source=12,
            maximum_attempts=8,
            pivot="de",
            batch_size=8,
            seed=seed,
            min_similarity=0.75,
            max_similarity=0.98,
            fallback_to_best=True,
        )

    raise ValueError(
        f"Unknown control_name: {control_name}"
    )

In [ ]:
def run_bt_control_experiment(
    control_name,
    seed=42,
    top_k=3,
    output_root=OUTPUT_ROOT,
):
    set_all_seeds(seed)

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    run_name = (
        f"phase4_control_{control_name}_"
        f"seed{seed}_{timestamp}"
    )

    out_dir = Path(output_root) / run_name

    if out_dir.exists():
        raise FileExistsError(
            f"Output folder already exists:\n{out_dir}"
        )

    out_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    print("=" * 80)
    print("PHASE 4 BACK-TRANSLATION CONTROL EXPERIMENT")
    print("=" * 80)
    print(f"Control: {control_name}")
    print(f"Seed:    {seed}")
    print(f"Output:  {out_dir}")
    print("=" * 80)

    train_df, val_df, test_df = load_data()

    original_training_size = len(train_df)

    original_class_counts = (
        train_df["ekman_id"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    print("\nOriginal dataset sizes:")
    print(f"Training:   {len(train_df):,}")
    print(f"Validation: {len(val_df):,}")
    print(f"Test:       {len(test_df):,}")

    print("\nOriginal training class counts:")
    print(
        train_df["ekman_id"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    augmentation_start = time.time()

    train_df, synthetic_df, augmentation_report = apply_augmentation_control(
        train_df=train_df,
        control_name=control_name,
        seed=seed,
        top_k=top_k,
    )

    augmentation_time = time.time() - augmentation_start

    print(
        f"\nAugmentation completed in "
        f"{augmentation_time:.1f} seconds "
        f"({augmentation_time / 60:.2f} minutes)."
    )

    print("\nFinal training class counts:")
    print(
        train_df["ekman_id"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    synthetic_df.to_csv(
        out_dir / "generated_synthetic_examples.csv",
        index=False,
    )

    with open(
        out_dir / "augmentation_report.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            make_json_safe(augmentation_report),
            file,
            indent=2,
            ensure_ascii=False,
        )


    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label={
            index: label
            for index, label in enumerate(EKMAN_LABELS)
        },
        label2id={
            label: index
            for index, label in enumerate(EKMAN_LABELS)
        },
    )

    train_ds = GoEmotionsDataset(
        train_df["text_clean"],
        train_df["ekman_id"],
        tokenizer,
    )

    val_ds = GoEmotionsDataset(
        val_df["text_clean"],
        val_df["ekman_id"],
        tokenizer,
    )

    test_ds = GoEmotionsDataset(
        test_df["text_clean"],
        test_df["ekman_id"],
        tokenizer,
    )

    training_args = TrainingArguments(
        output_dir=str(out_dir / "checkpoints"),
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=N_EPOCHS,
        warmup_steps=WARMUP_STEPS,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        logging_steps=100,
        seed=seed,
        data_seed=seed,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(
            tokenizer=tokenizer
        ),
        compute_metrics=hf_compute_metrics,
    )

    print("\nStarting training...")

    training_start = time.time()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    train_result = trainer.train()

    training_time = time.time() - training_start

    peak_gpu_memory_gb = None

    if torch.cuda.is_available():
        peak_gpu_memory_gb = (
            torch.cuda.max_memory_allocated() / 1e9
        )

    print(
        f"\nTraining completed in "
        f"{training_time:.1f} seconds "
        f"({training_time / 60:.2f} minutes)."
    )

    if peak_gpu_memory_gb is not None:
        print(
            f"Peak GPU memory: "
            f"{peak_gpu_memory_gb:.2f} GB"
        )

    best_model_dir = out_dir / "best_model"

    trainer.save_model(
        str(best_model_dir)
    )

    tokenizer.save_pretrained(
        str(best_model_dir)
    )


    def evaluate_and_save(
        dataset,
        dataframe,
        split_name,
    ):
        print(f"\nEvaluating {split_name} set...")

        prediction_output = trainer.predict(
            dataset,
            metric_key_prefix=split_name,
        )

        logits = prediction_output.predictions

        if isinstance(logits, tuple):
            logits = logits[0]

        y_true = prediction_output.label_ids.astype(int)
        y_pred = np.argmax(
            logits,
            axis=-1,
        ).astype(int)

        stable_logits = (
            logits
            - np.max(
                logits,
                axis=1,
                keepdims=True,
            )
        )

        exponentials = np.exp(stable_logits)

        probabilities = (
            exponentials
            / np.sum(
                exponentials,
                axis=1,
                keepdims=True,
            )
        )

        split_metrics = compute_metrics(
            y_true,
            y_pred,
            EKMAN_LABELS,
        )

        prediction_columns = {
            "text_clean": dataframe["text_clean"]
                .astype(str)
                .tolist(),
            "true_id": y_true,
            "true_label": [
                EKMAN_LABELS[value]
                for value in y_true
            ],
            "predicted_id": y_pred,
            "predicted_label": [
                EKMAN_LABELS[value]
                for value in y_pred
            ],
            "correct": y_true == y_pred,
        }

        if "id" in dataframe.columns:
            prediction_columns = {
                "id": dataframe["id"].tolist(),
                **prediction_columns,
            }

        predictions_dataframe = pd.DataFrame(
            prediction_columns
        )

        for class_index, class_name in enumerate(EKMAN_LABELS):
            safe_class_name = (
                str(class_name)
                .lower()
                .replace(" ", "_")
            )

            predictions_dataframe[
                f"probability_{safe_class_name}"
            ] = probabilities[:, class_index]

        predictions_dataframe.to_csv(
            out_dir / f"{split_name}_predictions.csv",
            index=False,
        )

        matrix = save_confusion_matrix_to_folder(
            y_true=y_true,
            y_pred=y_pred,
            labels=EKMAN_LABELS,
            output_path=(
                out_dir
                / f"{split_name}_confusion_matrix.png"
            ),
            title=f"{split_name.capitalize()} confusion matrix",
        )

        np.savetxt(
            out_dir / f"{split_name}_confusion_matrix.csv",
            matrix,
            delimiter=",",
            fmt="%d",
        )

        with open(
            out_dir / f"{split_name}_metrics.json",
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                make_json_safe(split_metrics),
                file,
                indent=2,
                ensure_ascii=False,
            )

        return split_metrics

    validation_metrics = evaluate_and_save(
        dataset=val_ds,
        dataframe=val_df,
        split_name="validation",
    )

    test_metrics = evaluate_and_save(
        dataset=test_ds,
        dataframe=test_df,
        split_name="test",
    )

    final_class_counts = (
        train_df["ekman_id"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    complete_results = {
        "run_name": run_name,
        "control_name": control_name,
        "seed": seed,
        "model_name": MODEL_NAME,
        "top_k": top_k,
        "original_training_size": int(original_training_size),
        "final_training_size": int(len(train_df)),
        "generated_synthetic_examples": int(len(synthetic_df)),
        "original_training_class_counts": {
            str(int(key)): int(value)
            for key, value in original_class_counts.items()
        },
        "final_training_class_counts": {
            str(int(key)): int(value)
            for key, value in final_class_counts.items()
        },
        "augmentation_time_seconds": augmentation_time,
        "training_time_seconds": training_time,
        "peak_gpu_memory_gb": peak_gpu_memory_gb,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_validation_macro_f1_during_training": trainer.state.best_metric,
        "augmentation": augmentation_report,
        "validation": validation_metrics,
        "test": test_metrics,
    }

    with open(
        out_dir / "complete_results.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            make_json_safe(complete_results),
            file,
            indent=2,
            ensure_ascii=False,
        )

    with open(
        out_dir / "training_log_history.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            make_json_safe(trainer.state.log_history),
            file,
            indent=2,
            ensure_ascii=False,
        )

    print("\n" + "=" * 80)
    print("FINAL RESULTS")
    print("=" * 80)

    print("\nValidation metrics:")
    print(
        json.dumps(
            make_json_safe(validation_metrics),
            indent=2,
        )
    )

    print("\nTest metrics:")
    print(
        json.dumps(
            make_json_safe(test_metrics),
            indent=2,
        )
    )

    print("\nAll results saved in:")
    print(out_dir)

    print("=" * 80)

    return complete_results

In [ ]:
CONTROL_EXPERIMENTS = [
    "direct_duplication_x3",

    "random_oversampling_x3",

    "bt_single_x1",

    "bt_diverse_x2",

    "bt_similarity_filtered_x3",
]

In [ ]:
def run_one_control_for_all_seeds(
    control_name,
    seeds=SEEDS,
    top_k=3,
    output_root=OUTPUT_ROOT,
):
    results = []

    print("=" * 80)
    print(f"Running control experiment: {control_name}")
    print(f"Seeds: {seeds}")
    print("=" * 80)

    for seed in seeds:
        print("\n" + "-" * 80)
        print(f"Starting {control_name} with seed {seed}")
        print("-" * 80)

        result = run_bt_control_experiment(
            control_name=control_name,
            seed=seed,
            top_k=top_k,
            output_root=output_root,
        )

        results.append(result)

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\n" + "=" * 80)
    print(f"Finished control experiment: {control_name}")
    print("=" * 80)

    return results

In [ ]:
direct_duplication_results = run_one_control_for_all_seeds(
    control_name="direct_duplication_x3",
)

Running control experiment: direct_duplication_x3
Seeds: [42, 0, 1]

--------------------------------------------------------------------------------
Starting direct_duplication_x3 with seed 42
--------------------------------------------------------------------------------
PHASE 4 BACK-TRANSLATION CONTROL EXPERIMENT
Control: direct_duplication_x3
Seed:    42
Output:  /content/drive/MyDrive/Thesis/results/phase4_bt_control_runs/phase4_control_direct_duplication_x3_seed42_20260701_155647

Original dataset sizes:
Training:   36,302
Validation: 4,547
Test:       4,590

Original training class counts:
ekman_id
0     3878
1      497
2      515
3    12920
4     2121
5     3553
6    12818

Augmentation completed in 0.3 seconds (0.01 minutes).

Final training class counts:
ekman_id
0     3878
1     1988
2     2060
3    12920
4     8484
5     3553
6    12818


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.802474,0.848429,0.689466,0.604835,0.689521
2,0.597936,0.871666,0.691885,0.607954,0.691791
3,0.473071,0.900271,0.691005,0.609546,0.691695


...


Training completed in 512.0 seconds (8.53 minutes).
Peak GPU memory: 3.28 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating validation set...



Evaluating test set...



FINAL RESULTS

Validation metrics:
{
  "accuracy": 0.6885858808005278,
  "macro_f1": 0.5997594104378342,
  "weighted_f1": 0.6894690641186463,
  "per_class_f1": [
    0.5155746509129968,
    0.43137254901960786,
    0.5874125874125874,
    0.8161676646706587,
    0.5950704225352113,
    0.586483390607102,
    0.6662346079066753
  ],
  "report": {
    "anger": {
      "precision": 0.5381165919282511,
      "recall": 0.4948453608247423,
      "f1-score": 0.5155746509129968,
      "support": 485.0
    },
    "disgust": {
      "precision": 0.358695652173913,
      "recall": 0.5409836065573771,
      "f1-score": 0.43137254901960786,
      "support": 61.0
    },
    "fear": {
      "precision": 0.5454545454545454,
      "recall": 0.6363636363636364,
      "f1-score": 0.5874125874125874,
      "support": 66.0
    },
    "joy": {
      "precision": 0.8151913875598086,
      "recall": 0.8171462829736211,
      "f1-score": 0.8161676646706587,
      "support": 1668.0
    },
    "sadness": {
    

In [ ]:
random_oversampling_results = run_one_control_for_all_seeds(
    control_name="random_oversampling_x3",
)

Running control experiment: random_oversampling_x3
Seeds: [42, 0, 1]

--------------------------------------------------------------------------------
Starting random_oversampling_x3 with seed 42
--------------------------------------------------------------------------------
PHASE 4 BACK-TRANSLATION CONTROL EXPERIMENT
Control: random_oversampling_x3
Seed:    42
Output:  /content/drive/MyDrive/Thesis/results/phase4_bt_control_runs/phase4_control_random_oversampling_x3_seed42_20260701_162330

Original dataset sizes:
Training:   36,302
Validation: 4,547
Test:       4,590

Original training class counts:
ekman_id
0     3878
1      497
2      515
3    12920
4     2121
5     3553
6    12818

Augmentation completed in 0.8 seconds (0.01 minutes).

Final training class counts:
ekman_id
0     3878
1     1988
2     2060
3    12920
4     8484
5     3553
6    12818


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.799255,0.861731,0.689466,0.610561,0.688367
2,0.592610,0.856977,0.691885,0.617430,0.691938
3,0.485626,0.888814,0.695623,0.619406,0.695985


...


Training completed in 512.2 seconds (8.54 minutes).
Peak GPU memory: 3.28 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating validation set...



Evaluating test set...



FINAL RESULTS

Validation metrics:
{
  "accuracy": 0.6863866285462943,
  "macro_f1": 0.6174015473932738,
  "weighted_f1": 0.690238129355832,
  "per_class_f1": [
    0.5096262740656852,
    0.5,
    0.6713286713286714,
    0.8054934525710635,
    0.5581395348837209,
    0.6,
    0.6772228989037758
  ],
  "report": {
    "anger": {
      "precision": 0.5653266331658291,
      "recall": 0.4639175257731959,
      "f1-score": 0.5096262740656852,
      "support": 485.0
    },
    "disgust": {
      "precision": 0.42528735632183906,
      "recall": 0.6065573770491803,
      "f1-score": 0.5,
      "support": 61.0
    },
    "fear": {
      "precision": 0.6233766233766234,
      "recall": 0.7272727272727273,
      "f1-score": 0.6713286713286714,
      "support": 66.0
    },
    "joy": {
      "precision": 0.8619275461380724,
      "recall": 0.7559952038369304,
      "f1-score": 0.8054934525710635,
      "support": 1668.0
    },
    "sadness": {
      "precision": 0.44554455445544555,
      "re

In [ ]:
bt_single_results = run_one_control_for_all_seeds(
    control_name="bt_single_x1",
)

Running control experiment: bt_single_x1
Seeds: [42, 0, 1]

--------------------------------------------------------------------------------
Starting bt_single_x1 with seed 42
--------------------------------------------------------------------------------
PHASE 4 BACK-TRANSLATION CONTROL EXPERIMENT
Control: bt_single_x1
Seed:    42
Output:  /content/drive/MyDrive/Thesis/results/phase4_bt_control_runs/phase4_control_bt_single_x1_seed42_20260701_164957

Original dataset sizes:
Training:   36,302
Validation: 4,547
Test:       4,590

Original training class counts:
ekman_id
0     3878
1      497
2      515
3    12920
4     2121
5     3553
6    12818

Loading Helsinki-NLP/opus-mt-en-de...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]


Translating English to pivot language...

Loading Helsinki-NLP/opus-mt-de-en...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

...


Training completed in 435.7 seconds (7.26 minutes).
Peak GPU memory: 3.28 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating validation set...



Evaluating test set...



FINAL RESULTS

Validation metrics:
{
  "accuracy": 0.6927644600835716,
  "macro_f1": 0.628244166058883,
  "weighted_f1": 0.692784096790392,
  "per_class_f1": [
    0.544891640866873,
    0.48120300751879697,
    0.7058823529411765,
    0.814508994396933,
    0.5949820788530465,
    0.597574421168688,
    0.6586666666666666
  ],
  "report": {
    "anger": {
      "precision": 0.5454545454545454,
      "recall": 0.5443298969072164,
      "f1-score": 0.544891640866873,
      "support": 485.0
    },
    "disgust": {
      "precision": 0.4444444444444444,
      "recall": 0.5245901639344263,
      "f1-score": 0.48120300751879697,
      "support": 61.0
    },
    "fear": {
      "precision": 0.6857142857142857,
      "recall": 0.7272727272727273,
      "f1-score": 0.7058823529411765,
      "support": 66.0
    },
    "joy": {
      "precision": 0.8015089959373186,
      "recall": 0.8279376498800959,
      "f1-score": 0.814508994396933,
      "support": 1668.0
    },
    "sadness": {
      "pr

In [ ]:
bt_diverse_x2_results = run_one_control_for_all_seeds(
    control_name="bt_diverse_x2",
)

Running control experiment: bt_diverse_x2
Seeds: [42, 0, 1]

--------------------------------------------------------------------------------
Starting bt_diverse_x2 with seed 42
--------------------------------------------------------------------------------
PHASE 4 BACK-TRANSLATION CONTROL EXPERIMENT
Control: bt_diverse_x2
Seed:    42
Output:  /content/drive/MyDrive/Thesis/results/phase4_bt_control_runs/phase4_control_bt_diverse_x2_seed42_20260701_175636

Original dataset sizes:
Training:   36,302
Validation: 4,547
Test:       4,590

Original training class counts:
ekman_id
0     3878
1      497
2      515
3    12920
4     2121
5     3553
6    12818

Loading Helsinki-NLP/opus-mt-en-de...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]


Translating English to pivot language...

Loading Helsinki-NLP/opus-mt-de-en...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

...


Training completed in 479.8 seconds (8.00 minutes).
Peak GPU memory: 3.28 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating validation set...



Evaluating test set...



FINAL RESULTS

Validation metrics:
{
  "accuracy": 0.6830877501649439,
  "macro_f1": 0.5980178376218055,
  "weighted_f1": 0.6829180185967589,
  "per_class_f1": [
    0.4774774774774775,
    0.4714285714285714,
    0.647887323943662,
    0.8133619368679129,
    0.5483870967741935,
    0.5534591194968553,
    0.6741233373639661
  ],
  "report": {
    "anger": {
      "precision": 0.5260545905707196,
      "recall": 0.43711340206185567,
      "f1-score": 0.4774774774774775,
      "support": 485.0
    },
    "disgust": {
      "precision": 0.4177215189873418,
      "recall": 0.5409836065573771,
      "f1-score": 0.4714285714285714,
      "support": 61.0
    },
    "fear": {
      "precision": 0.6052631578947368,
      "recall": 0.696969696969697,
      "f1-score": 0.647887323943662,
      "support": 66.0
    },
    "joy": {
      "precision": 0.831974921630094,
      "recall": 0.7955635491606715,
      "f1-score": 0.8133619368679129,
      "support": 1668.0
    },
    "sadness": {
      "

In [ ]:
bt_similarity_filtered_results = run_one_control_for_all_seeds(
    control_name="bt_similarity_filtered_x3",
)

Running control experiment: bt_similarity_filtered_x3
Seeds: [42, 0, 1]

--------------------------------------------------------------------------------
Starting bt_similarity_filtered_x3 with seed 42
--------------------------------------------------------------------------------
PHASE 4 BACK-TRANSLATION CONTROL EXPERIMENT
Control: bt_similarity_filtered_x3
Seed:    42
Output:  /content/drive/MyDrive/Thesis/results/phase4_bt_control_runs/phase4_control_bt_similarity_filtered_x3_seed42_20260701_191238

Original dataset sizes:
Training:   36,302
Validation: 4,547
Test:       4,590

Original training class counts:
ekman_id
0     3878
1      497
2      515
3    12920
4     2121
5     3553
6    12818

Loading Helsinki-NLP/opus-mt-en-de...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]


Translating English to pivot language...

Loading Helsinki-NLP/opus-mt-de-en...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

...


Training completed in 497.4 seconds (8.29 minutes).
Peak GPU memory: 3.28 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating validation set...



Evaluating test set...



FINAL RESULTS

Validation metrics:
{
  "accuracy": 0.674510666373433,
  "macro_f1": 0.595195001408575,
  "weighted_f1": 0.6795471296846652,
  "per_class_f1": [
    0.4845005740528129,
    0.4311377245508982,
    0.6762589928057554,
    0.8082148499210111,
    0.5227606461086637,
    0.5782857142857143,
    0.6652065081351689
  ],
  "report": {
    "anger": {
      "precision": 0.5466321243523317,
      "recall": 0.4350515463917526,
      "f1-score": 0.4845005740528129,
      "support": 485.0
    },
    "disgust": {
      "precision": 0.33962264150943394,
      "recall": 0.5901639344262295,
      "f1-score": 0.4311377245508982,
      "support": 61.0
    },
    "fear": {
      "precision": 0.6438356164383562,
      "recall": 0.7121212121212122,
      "f1-score": 0.6762589928057554,
      "support": 66.0
    },
    "joy": {
      "precision": 0.85437541750167,
      "recall": 0.7667865707434053,
      "f1-score": 0.8082148499210111,
      "support": 1668.0
    },
    "sadness": {
      "

In [ ]:
def collect_control_results(output_root):
    output_root = Path(output_root)

    rows = []

    for result_file in output_root.glob(
        "phase4_control_*/complete_results.json"
    ):
        with open(
            result_file,
            "r",
            encoding="utf-8",
        ) as file:
            result = json.load(file)

        row = {
            "run_name": result["run_name"],
            "control_name": result["control_name"],
            "seed": result["seed"],
            "generated_synthetic_examples": result[
                "generated_synthetic_examples"
            ],
            "final_training_size": result[
                "final_training_size"
            ],
            "best_validation_macro_f1": result[
                "best_validation_macro_f1_during_training"
            ],
            "validation_accuracy": result[
                "validation"
            ]["accuracy"],
            "validation_macro_f1": result[
                "validation"
            ]["macro_f1"],
            "validation_weighted_f1": result[
                "validation"
            ]["weighted_f1"],
            "test_accuracy": result[
                "test"
            ]["accuracy"],
            "test_macro_f1": result[
                "test"
            ]["macro_f1"],
            "test_weighted_f1": result[
                "test"
            ]["weighted_f1"],
        }

        augmentation = result.get(
            "augmentation",
            {},
        )

        row["selected_class_ids"] = augmentation.get(
            "selected_class_ids"
        )

        row["mean_semantic_similarity"] = augmentation.get(
            "mean_semantic_similarity"
        )

        row["passed_similarity_filter_count"] = augmentation.get(
            "passed_similarity_filter_count"
        )

        row["sources_with_insufficient_passing_candidates"] = augmentation.get(
            "sources_with_insufficient_passing_candidates"
        )

        rows.append(row)

    summary_df = pd.DataFrame(rows)

    if len(summary_df) == 0:
        raise FileNotFoundError(
            f"No complete_results.json files found under {output_root}"
        )

    summary_df = summary_df.sort_values(
        [
            "control_name",
            "seed",
        ]
    ).reset_index(drop=True)

    return summary_df


control_summary = collect_control_results(
    OUTPUT_ROOT
)

control_summary.to_csv(
    OUTPUT_ROOT / "phase4_bt_control_summary_by_seed.csv",
    index=False,
)

control_summary

,run_name,control_name,seed,generated_synthetic_examples,final_training_size,best_validation_macro_f1,validation_accuracy,validation_macro_f1,validation_weighted_f1,test_accuracy,test_macro_f1,test_weighted_f1,selected_class_ids,mean_semantic_similarity,passed_similarity_filter_count,sources_with_insufficient_passing_candidates
0,phase4_control_bt_diverse_x2_seed0_20260701_18...,bt_diverse_x2,0,6266,42568,0.613085,0.691665,0.613085,0.694087,0.682135,0.619636,0.682926,"[1, 2, 4]",NaN,NaN,NaN
1,phase4_control_bt_diverse_x2_seed1_20260701_18...,bt_diverse_x2,1,6266,42568,0.598018,0.683088,0.598018,0.682918,0.685621,0.610729,0.684296,"[1, 2, 4]",NaN,NaN,NaN
2,phase4_control_bt_diverse_x2_seed42_20260701_1...,bt_diverse_x2,42,6266,42568,0.612767,0.692325,0.612767,0.693594,0.686492,0.621646,0.686887,"[1, 2, 4]",NaN,NaN,NaN
3,phase4_control_bt_similarity_filtered_x3_seed0...,bt_similarity_filtered_x3,0,9399,45701,0.606283,0.682208,0.606283,0.685192,0.673638,0.611634,0.676202,"[1, 2, 4]",0.874564,8106.0,520.0
4,phase4_control_bt_similarity_filtered_x3_seed1...,bt_similarity_filtered_x3,1,9399,45701,0.595195,0.674511,0.595195,0.679547,0.667102,0.611238,0.671650,"[1, 2, 4]",0.874653,8114.0,520.0
5,phase4_control_bt_similarity_filtered_x3_seed4...,bt_similarity_filtered_x3,42,9399,45701,0.587223,0.681548,0.587223,0.679734,0.685185,0.609230,0.683239,"[1, 2, 4]",0.874537,8125.0,520.0
6,phase4_control_bt_single_x1_seed0_20260701_171219,bt_single_x1,0,3133,39435,0.623372,0.697823,0.623372,0.696960,0.690414,0.626231,0.687959,"[1, 2, 4]",NaN,NaN,NaN
7,phase4_control_bt_single_x1_seed1_20260701_173424,bt_single_x1,1,3133,39435,0.628244,0.692764,0.628244,0.692784,0.684314,0.624553,0.683366,"[1, 2, 4]",NaN,NaN,NaN
8,phase4_control_bt_single_x1_seed42_20260701_16...,bt_single_x1,42,3133,39435,0.624623,0.689246,0.624623,0.689236,0.691068,0.614899,0.689954,"[1, 2, 4]",NaN,NaN,NaN
9,phase4_control_direct_duplication_x3_seed0_202...,direct_duplication_x3,0,9399,45701,0.612440,0.685727,0.612440,0.686290,0.676035,0.618182,0.675187,"[1, 2, 4]",NaN,NaN,NaN


In [ ]:
summary_by_control = (
    control_summary
    .groupby("control_name")
    .agg(
        seeds=("seed", "count"),
        generated_synthetic_examples_mean=(
            "generated_synthetic_examples",
            "mean",
        ),
        validation_macro_f1_mean=(
            "validation_macro_f1",
            "mean",
        ),
        validation_macro_f1_std=(
            "validation_macro_f1",
            "std",
        ),
        test_macro_f1_mean=(
            "test_macro_f1",
            "mean",
        ),
        test_macro_f1_std=(
            "test_macro_f1",
            "std",
        ),
        test_accuracy_mean=(
            "test_accuracy",
            "mean",
        ),
        test_accuracy_std=(
            "test_accuracy",
            "std",
        ),
        mean_semantic_similarity=(
            "mean_semantic_similarity",
            "mean",
        ),
        passed_similarity_filter_count_mean=(
            "passed_similarity_filter_count",
            "mean",
        ),
    )
    .reset_index()
)

summary_by_control.to_csv(
    OUTPUT_ROOT / "phase4_bt_control_summary_mean_std.csv",
    index=False,
)

summary_by_control

,control_name,seeds,generated_synthetic_examples_mean,validation_macro_f1_mean,validation_macro_f1_std,test_macro_f1_mean,test_macro_f1_std,test_accuracy_mean,test_accuracy_std,mean_semantic_similarity,passed_similarity_filter_count_mean
0,bt_diverse_x2,3,6266.0,0.607957,0.008609,0.617337,0.005810,0.684749,0.002306,NaN,NaN
1,bt_similarity_filtered_x3,3,9399.0,0.596234,0.009572,0.610700,0.001289,0.675309,0.009156,0.874584,8115.0
2,bt_single_x1,3,3133.0,0.625413,0.002530,0.621894,0.006116,0.688598,0.003725,NaN,NaN
3,direct_duplication_x3,6,9399.0,0.607248,0.005944,0.614092,0.003172,0.681772,0.004454,NaN,NaN
4,random_oversampling_x3,6,9399.0,0.618381,0.000897,0.617037,0.000814,0.681772,0.003320,NaN,NaN
